In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, accuracy_score, recall_score, f1_score

train_df = pd.read_csv('train_preprocessed.csv')
test_df = pd.read_csv('test_preprocessed.csv')

X_train = train_df.drop('RainTomorrow', axis=1)
y_train = train_df['RainTomorrow']
X_test = test_df.drop('RainTomorrow', axis=1)
y_test = test_df['RainTomorrow']

X_tune = X_train.sample(frac=1.0, random_state=42)
y_tune = y_train.loc[X_tune.index]

knn_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('knn', KNeighborsClassifier(n_jobs=-1))
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 11, 15, 21],
    'knn__weights': ['uniform', 'distance']
}

scoring_metrics = {'f1': 'f1', 'accuracy' : 'accuracy', 'precision': 'precision', 'recall': 'recall'}

search = GridSearchCV(
    knn_pipeline, 
    param_grid=param_grid, 
    cv=3, 
    scoring=scoring_metrics,
    refit='f1',
    n_jobs=-1
)
search.fit(X_tune, y_tune)

results = pd.DataFrame(search.cv_results_)
cols = [
    'param_knn__n_neighbors', 'param_knn__weights', 'mean_test_accuracy', 
    'mean_test_precision', 'mean_test_recall', 'mean_test_f1', 'rank_test_f1'
]
report_df = results[cols].sort_values(by='rank_test_f1')
report_df.columns = ['n_neighbors', 'weights', 'avg_Accuracy', 'avg_Precision', 'avg_Recall', 'avg_F1-Score', 'Rank']

for col in ['avg_Accuracy', 'avg_Precision', 'avg_Recall', 'avg_F1-Score']:
    report_df[col] = report_df[col].round(4)

print(report_df.to_string(index=False))
best_knn = search.best_estimator_
best_knn.fit(X_train, y_train)

y_pred = best_knn.predict(X_test)
y_pred_proba = best_knn.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:     {accuracy:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1-Score:      {f1:.4f}")

auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {auc_score:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results_file = 'model_results.csv'

new_result = pd.DataFrame([{
    'Model': 'KNN',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1': f1,
    'ROC_AUC': auc_score
}])

if os.path.exists(results_file):
    existing_df = pd.read_csv(results_file)
else:
    existing_df = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC'])

existing_df = existing_df[existing_df['Model'] != 'KNN']
updated_df = pd.concat([existing_df, new_result], ignore_index=True)
updated_df.to_csv(results_file, index=False)

print("\nUpdated model_results.csv:")
print(updated_df)

 n_neighbors  weights  avg_Accuracy  avg_Precision  avg_Recall  avg_F1-Score  Rank
           3 distance        0.8540         0.7828      0.9798        0.8703     1
           3  uniform        0.8477         0.7800      0.9686        0.8642     2
           5 distance        0.8443         0.7718      0.9777        0.8627     3
           7 distance        0.8395         0.7670      0.9753        0.8587     4
          11 distance        0.8354         0.7640      0.9706        0.8550     5
           5  uniform        0.8337         0.7673      0.9581        0.8521     6
          15 distance        0.8319         0.7620      0.9654        0.8517     7
          21 distance        0.8274         0.7596      0.9581        0.8474     8
           7  uniform        0.8261         0.7613      0.9500        0.8453     9
          11  uniform        0.8175         0.7566      0.9363        0.8369    10
          15  uniform        0.8121         0.7538      0.9270        0.8315    11
    